First of all we need to install requied **libraries**

In [21]:
!pip install transformers datasets peft evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00


Verifying Installations

In [22]:
import transformers
import datasets
import peft
import evaluate
print(f"Tramsformers: {transformers.__version__}")
print(f"Datasets: {datasets.__version__}")
print(f"PEFT: {peft.__version__}")
print(f"Evaluate: {evaluate.__version__}")

Tramsformers: 5.0.0
Datasets: 4.0.0
PEFT: 0.18.1
Evaluate: 0.4.6


**Load the datasets**

In [18]:
from datasets import load_dataset

loading banking77 dataset

In [19]:
dataset =load_dataset("legacy-datasets/banking77")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/298k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/93.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

In [23]:
print(dataset)
print("\nfirst training example:")
print(dataset["train"][0])
print("\nNumber of classes:")
print(dataset["train"].features["label"].num_classes)
print("\nClass name sample:")
print(dataset["train"].features["label"].names[:10])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})

first training example:
{'text': 'I am still waiting on my card?', 'label': 11}

Number of classes:
77

Class name sample:
['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire']


Exploration of our data

In [24]:
import pandas as pd
from collections import Counter

# Convert to pandas for easy exploration
train_df = pd.DataFrame(dataset["train"])

# Check class distribution
label_counts = Counter(train_df["label"])
print(f"Most common class count: {max(label_counts.values())}")
print(f"Least common class count: {min(label_counts.values())}")
print(f"Average examples per class: {len(train_df) / 77:.1f}")

# Check text lengths
train_df["text_length"] = train_df["text"].apply(len)
print(f"\nAverage text length: {train_df['text_length'].mean():.1f} chars")
print(f"Max text length: {train_df['text_length'].max()} chars")
print(f"Min text length: {train_df['text_length'].min()} chars")

# Sample a few examples
print("\nSample examples:")
print(train_df.sample(3)[["text", "label"]].to_string())

Most common class count: 187
Least common class count: 35
Average examples per class: 129.9

Average text length: 59.5 chars
Max text length: 433 chars
Min text length: 13 chars

Sample examples:
                                          text  label
6323     What happens when I am charged twice?     63
7588     I need to deposit money to my account     65
8413  I need to get a disposable virtual card?     37


Load Tokenizer

In [13]:
from transformers import AutoTokenizer

# Load DistilBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Test it on a sample
sample_text = "I am still waiting on my card"
tokens = tokenizer(sample_text, return_tensors="pt")

print(f"Input IDs: {tokens['input_ids']}")
print(f"Attention mask: {tokens['attention_mask']}")
print(f"Number of tokens: {tokens['input_ids'].shape[1]}")

# Decode back to see what tokenization looks like
decoded = tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])
print(f"\nTokens: {decoded}")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Input IDs: tensor([[ 101, 1045, 2572, 2145, 3403, 2006, 2026, 4003,  102]])
Attention mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])
Number of tokens: 9

Tokens: ['[CLS]', 'i', 'am', 'still', 'waiting', 'on', 'my', 'card', '[SEP]']


Tokenizing full dataset

In [25]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Apply tokenization to entire dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Format for PyTorch
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch",
    columns=["input_ids", "attention_mask", "labels"])

print("Tokenization complete")
print(tokenized_dataset)

Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

Tokenization complete
DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3080
    })
})


Next we load D.Bert model

In [8]:
from transformers import AutoModelForSequenceClassification
import torch

# Load DistilBERT with classification head
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=77
)

# Check model size
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters: 67,012,685
Trainable parameters: 67,012,685

Model architecture:
DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affin

Apply LORA

In [10]:
from peft import LoraConfig, get_peft_model, TaskType

# Define LoRA configuration
lora_config = LoraConfig(
    r=8,                     # rank
    lora_alpha=16,           # scaling factor
    target_modules=["q_lin", "v_lin"],  # attention matrices
    lora_dropout=0.1,        # dropout for regularization
    bias="none",             # don't train bias terms
    task_type=TaskType.SEQ_CLS  # sequence classification
)

# Wrap model with LoRA
model = get_peft_model(model, lora_config)

# Verify trainable parameters
model.print_trainable_parameters()

trainable params: 797,261 || all params: 67,809,946 || trainable%: 1.1757


Handle Class Imbalance

In [26]:
import torch
import numpy as np
from torch.nn import CrossEntropyLoss

# Calculate class weights
label_counts = torch.zeros(77)
for example in tokenized_dataset["train"]:
    label_counts[example["labels"]] += 1

# Inverse frequency weighting
class_weights = 1.0 / label_counts
class_weights = class_weights / class_weights.sum() * 77

print("Class weights calculated")
print(f"Max weight: {class_weights.max():.4f}")
print(f"Min weight: {class_weights.min():.4f}")
print(f"Weight ratio: {class_weights.max()/class_weights.min():.2f}x")

Class weights calculated
Max weight: 3.3678
Min weight: 0.6303
Weight ratio: 5.34x


Training Setup

In [27]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

# Load accuracy metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Training arguments
training_args = TrainingArguments(
    output_dir="./banking77-lora",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True
)

print("Training arguments configured")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments configured
Epochs: 5
Batch size: 32


 Create Weighted Trainer

In [28]:
from torch.nn import CrossEntropyLoss

# Custom trainer to handle class imbalance
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Apply class weights to loss
        loss_fct = CrossEntropyLoss(
            weight=class_weights.to(model.device)
        )
        loss = loss_fct(logits.view(-1, 77), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# Split validation set from training data
train_val = tokenized_dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

# Initialize trainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_val["train"],
    eval_dataset=train_val["test"],
    compute_metrics=compute_metrics,
)

print("Trainer initialized")
print(f"Training examples: {len(train_val['train'])}")
print(f"Validation examples: {len(train_val['test'])}")

Trainer initialized
Training examples: 9002
Validation examples: 1001


Training of model

In [29]:
# Save model to Google Drive to prevent losing progress
from google.colab import drive
drive.mount('/content/drive')

# Start training
print("Starting training...")
trainer.train()

# Save final model
trainer.save_model("/content/drive/MyDrive/banking77-lora-final")
print("Model saved to Google Drive")

Mounted at /content/drive
Starting training...


Epoch,Training Loss,Validation Loss,Accuracy
1,3.995700,3.609172,0.293706
2,2.572789,2.307555,0.604396
3,2.008083,1.736333,0.693307
4,1.693004,1.496839,0.712288
5,1.582025,1.425961,0.735265


Model saved to Google Drive


Model Evaluation

In [30]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Get predictions on test set
predictions = trainer.predict(tokenized_dataset["test"])
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

# Overall accuracy
accuracy = (pred_labels == true_labels).mean()
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Detailed per class report
class_names = dataset["test"].features["label"].names
report = classification_report(
    true_labels,
    pred_labels,
    target_names=class_names,
    output_dict=True
)

# Find best and worst performing intents
report_df = pd.DataFrame(report).transpose()
report_df = report_df.drop(
    ["accuracy", "macro avg", "weighted avg"]
)

print("\nTop 5 best performing intents:")
print(report_df.nlargest(5, "f1-score")[["f1-score", "support"]])

print("\nTop 5 worst performing intents:")
print(report_df.nsmallest(5, "f1-score")[["f1-score", "support"]])

Test Accuracy: 0.7175 (71.75%)

Top 5 best performing intents:
                    f1-score  support
verify_top_up       1.000000     40.0
age_limit           0.975610     40.0
change_pin          0.950000     40.0
passcode_forgotten  0.930233     40.0
get_physical_card   0.928571     40.0

Top 5 worst performing intents:
                                    f1-score  support
transfer_not_received_by_recipient  0.310345     40.0
cash_withdrawal_not_recognised      0.338983     40.0
supported_cards_and_currencies      0.339623     40.0
topping_up_by_card                  0.339623     40.0
top_up_by_bank_transfer_charge      0.392857     40.0


In [32]:
model.save_pretrained("banking77-lora-final")
tokenizer.save_pretrained("banking77-lora-final")
print("Model saved locally")

Model saved locally


Inference Code

In [33]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# Load your fine-tuned model
def load_model(model_path="./banking77-lora-final"):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    config = PeftConfig.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(
        config.base_model_name_or_path,
        num_labels=77
    )
    model = PeftModel.from_pretrained(model, model_path)
    model.eval()
    return model, tokenizer

# Predict intent
def predict_intent(text, model, tokenizer):
    class_names = dataset["test"].features["label"].names

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=-1)
        confidence, predicted_class = torch.max(probabilities, dim=-1)

    return {
        "intent": class_names[predicted_class.item()],
        "confidence": f"{confidence.item()*100:.2f}%",
        "text": text
    }

# Load model
model, tokenizer = load_model()

# Test with real examples
test_queries = [
    "I lost my card and need a replacement",
    "Why was my payment declined?",
    "How do I add money to my account?",
    "I want to change my PIN number",
    "What currencies do you support?"
]

print("Intent Classification Results:")
print("="*50)
for query in test_queries:
    result = predict_intent(query, model, tokenizer)
    print(f"\nText: {result['text']}")
    print(f"Intent: {result['intent']}")
    print(f"Confidence: {result['confidence']}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Intent Classification Results:

Text: I lost my card and need a replacement
Intent: card_not_working
Confidence: 14.40%

Text: Why was my payment declined?
Intent: declined_card_payment
Confidence: 19.94%

Text: How do I add money to my account?
Intent: transfer_into_account
Confidence: 16.11%

Text: I want to change my PIN number
Intent: change_pin
Confidence: 47.46%

Text: What currencies do you support?
Intent: fiat_currency_support
Confidence: 35.15%
